# Waste Classification Model Training

Run this notebook in Google Colab with GPU enabled (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!pip install -q kaggle tensorflow scikit-learn

In [ ]:
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Enter your KAGGLE_API_TOKEN: ')

In [ ]:
!kaggle datasets download -d techsash/waste-classification-data
!unzip -q waste-classification-data.zip -d waste_dataset
!ls waste_dataset/DATASET/TRAIN

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import json
import os

# Setup paths and data generators
train_dir = "waste_dataset/DATASET/TRAIN"
test_dir = "waste_dataset/DATASET/TEST"

# Sometimes Kaggle dataset extraction creates a nested dataset structure based on how you unzip it
if not os.path.exists(train_dir) and os.path.exists("waste_dataset/dataset/DATASET/TRAIN"):
    train_dir = "waste_dataset/dataset/DATASET/TRAIN"
    test_dir = "waste_dataset/dataset/DATASET/TEST"

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='training'
)
val_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='validation'
)

# Build Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
model.fit(train_generator, validation_data=val_generator, epochs=5)

# Save Model
model.save("waste_model.h5")

# Save class label mapping
class_map = {str(i): name for name, i in train_generator.class_indices.items()}
with open("waste_classes.json", "w") as f:
    json.dump(class_map, f, indent=2)


In [ ]:
from google.colab import files
files.download('waste_model.h5')
files.download('waste_classes.json')